# Fine-Tuning Genomic Models for Sequence Classification

**Day 2 Morning - Session 1**

**Author:** Ikram Ullah, KAUST Bioinformatics Platform

---

## Overview

In this notebook, we'll **fine-tune** a pre-trained Nucleotide Transformer for a real genomic task: **promoter detection**.

This is the complete workflow you'll use in your research:

1. Load a pre-trained foundation model
2. Add a classification head
3. Prepare and tokenize your dataset
4. Train using Hugging Face Trainer
5. Evaluate on held-out test data

---

## Learning Objectives

1. Build a classifier on top of a pre-trained encoder
2. Use the Hugging Face `Trainer` for training
3. Implement proper evaluation metrics
4. Interpret results with confusion matrices

---

## The Task: Promoter Detection

**Promoters** are DNA regions that initiate gene transcription.

| Aspect | Details |
|--------|--------|
| **Input** | 300 nucleotide DNA sequence |
| **Output** | Binary: promoter (1) or non-promoter (0) |
| **Why it matters** | Understanding gene regulation, synthetic biology |

---

## Recap: What We've Learned

From Day 1:
- ✅ DNA encoding and tokenization (BPE, k-mer)
- ✅ Model architecture (Nucleotide Transformer)
- ✅ Embedding extraction
- ✅ Hugging Face ecosystem

Today we put it all together!

---

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    TrainingArguments, 
    Trainer
)
from datasets import load_dataset, DatasetDict
import evaluate
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {device}")

---

## Part 1: Load and Explore the Dataset

We'll use InstaDeep's benchmark dataset with 18 genomic tasks.

In [ ]:
# Load the full dataset
print("Loading dataset from Hugging Face Hub...")
ds_all = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
print(ds_all)

In [ ]:
# List available tasks
available_tasks = np.unique(ds_all['train']['task'])
print(f"Available tasks ({len(available_tasks)}):")
for task in available_tasks:
    print(f"  - {task}")

In [ ]:
# Select the promoter detection task
TASK = "promoter_all"  # Try others: "enhancers", "H3K4me3", "splice_sites_all"

# Filter to selected task
ds_full = ds_all.filter(lambda ex: ex["task"] == TASK)

# Create a subset for faster training (remove for full training)
ds = DatasetDict({
    "train": ds_full["train"].shuffle(seed=42).select(range(15000)),
    "test": ds_full["test"].shuffle(seed=42).select(range(5000))
})

print(f"Task: {TASK}")
print(f"Train samples: {len(ds['train']):,}")
print(f"Test samples: {len(ds['test']):,}")

In [ ]:
# Explore the data
print("Features:", ds['train'].features)

# Class distribution
labels = [ex['label'] for ex in ds['train']]
label_counts = Counter(labels)
print(f"\nClass distribution (training):")
for label, count in sorted(label_counts.items()):
    print(f"  Label {label}: {count:,} ({100*count/len(labels):.1f}%)")

# Sequence length
seq_lengths = [len(ex['sequence']) for ex in ds['train']]
print(f"\nSequence length: {min(seq_lengths)} - {max(seq_lengths)} bp")

# Sample
print(f"\nSample sequence (first 50bp): {ds['train'][0]['sequence'][:50]}...")

---

## Part 2: Tokenize the Dataset

Convert DNA strings to token IDs using the model's tokenizer.

In [ ]:
# Model for tokenization (use smaller model for speed)
MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Max sequence length (in tokens)
MAX_LEN = 512

def preprocess(batch):
    """Tokenize DNA sequences and add labels."""
    tokens = tokenizer(
        batch["sequence"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    tokens["labels"] = batch["label"]
    return tokens

# Apply tokenization
print("Tokenizing...")
tokenized = DatasetDict({
    k: v.map(preprocess, batched=True, remove_columns=v.column_names)
    for k, v in ds.items()
})

# Create validation split
split = tokenized["train"].train_test_split(test_size=0.1, seed=42)
tokenized = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": tokenized["test"]
})

print(f"\nTokenized dataset:")
print(tokenized)

---

## Part 3: Build the Classifier

We'll create a custom classifier that:
1. Uses the pre-trained NT encoder as backbone
2. Adds a linear classification head on top

### Architecture

```
Input IDs → NT Encoder → [CLS] Token → Linear Layer → Class Logits
            (frozen or   (hidden_dim)  (num_classes)
             trainable)
```

In [ ]:
# Use larger model for better performance
MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-100m-multi-species"

# Load pre-trained encoder
print(f"Loading {MODEL_ID}...")
base_model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Get dimensions
hidden_size = base_model.config.hidden_size
num_labels = len(set(ds["train"]["label"]))

print(f"Hidden size: {hidden_size}")
print(f"Number of classes: {num_labels}")

In [ ]:
class NTClassifier(nn.Module):
    """
    Sequence classifier using Nucleotide Transformer as backbone.
    
    Architecture:
        Input → NT Encoder → CLS token → Linear → Logits
    """
    
    def __init__(self, base_model, num_labels, hidden_size):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Pool: use CLS token (first token)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        
        # Classification head
        logits = self.classifier(pooled_output)
        
        # Compute loss if labels provided
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        return {"loss": loss, "logits": logits}

# Create model
model = NTClassifier(base_model, num_labels, hidden_size)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel created!")
print(f"  Total parameters:     {total_params/1e6:.1f}M")
print(f"  Trainable parameters: {trainable_params/1e6:.1f}M")

---

## Part 4: Train the Model

We'll use the Hugging Face `Trainer` which handles:
- Batching and data loading
- Forward/backward passes
- Optimizer (AdamW)
- Learning rate scheduling
- Logging and evaluation

In [ ]:
# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy and F1 for evaluation."""
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
# Training configuration
args = TrainingArguments(
    output_dir="nt_promoter_classifier",
    
    # Batch size (reduce if OOM)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    # Learning rate
    learning_rate=2e-5,
    
    # Training duration
    num_train_epochs=2,
    
    # Evaluation
    eval_strategy="epoch",
    
    # Logging
    logging_steps=50,
    
    # Checkpointing
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    
    # Disable external logging
    report_to="none",
)

print("Training configuration:")
print(f"  Batch size: {args.per_device_train_batch_size}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Epochs: {args.num_train_epochs}")

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train!
print("Starting training...")
print("="*50)
train_result = trainer.train()

# Print training results
print("\n" + "="*50)
print("Training complete!")
print(f"  Training loss: {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on validation set
print("Validation Results:")
print("="*50)
val_results = trainer.evaluate()
for k, v in val_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

---

## Part 5: Evaluate on Test Set

Final evaluation on held-out test data.

In [ ]:
# Predict on test set
print("Evaluating on test set...")
predictions = trainer.predict(tokenized["test"])

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Classification report
print("\n" + "="*50)
print("Classification Report")
print("="*50)
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"\n  Rows = true labels, Columns = predicted labels")
print(f"  Correct predictions: {cm.diagonal().sum()}/{len(y_true)} ({100*cm.diagonal().sum()/len(y_true):.1f}%)")

# Plot confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - {TASK}')
for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout()
plt.show()

---

## Exercises: Experiment!

### 1. Try a Different Task
Change `TASK = "promoter_all"` to:
- `"enhancers"` - Enhancer detection
- `"H3K4me3"` - Histone modification
- `"splice_sites_all"` - Splice site prediction

### 2. Tune Hyperparameters
- Learning rate: `1e-5`, `5e-5`, `3e-4`
- Batch size: `8`, `32`, `64`
- Epochs: `3`, `5`, `10`

### 3. Try a Larger Model
- `InstaDeepAI/nucleotide-transformer-500m-human-ref`

### 4. Freeze the Encoder
Only train the classification head:
```python
for param in model.base_model.parameters():
    param.requires_grad = False
```

---

## Summary

In this notebook, you learned the complete fine-tuning workflow:

| Step | What You Did |
|------|-------------|
| 1. Data | Loaded and explored benchmark dataset |
| 2. Tokenize | Converted DNA to token IDs |
| 3. Model | Built classifier with pre-trained backbone |
| 4. Train | Used HF Trainer for training loop |
| 5. Evaluate | Assessed with accuracy, F1, confusion matrix |

**Next**: We'll learn **PEFT/LoRA** for parameter-efficient fine-tuning (training only ~1% of parameters)!